<a name='0'></a>
## Packages

In [ ]:
from keras.layers import Bidirectional, Concatenate, Permute, Dot, Input, LSTM, Multiply
from keras.layers import RepeatVector, Dense, Activation, Lambda
from keras.optimizers import Adam
from keras.utils import to_categorical
from keras.models import load_model, Model
import keras.backend as K
import tensorflow as tf
import numpy as np
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
import pickle
from keras.layers import Softmax
import random
from babel.dates import format_date
import matplotlib.pyplot as plt
%matplotlib inline

<a name='1'></a>
### 1 - Dataset e pré processamento

In [ ]:

def load_dataset(m):
    """
    Carrega e retorna um subconjunto dos dados e vocabulários do projeto.

    A função abre os arquivos estáticos '.pkl' localizados na pasta 'data/', 
    lê o dataset principal e os dicionários de mapeamento, e retorna
    o dataset limitado aos primeiros 'm' exemplos.

    Args:
        m (int): O número de exemplos do dataset a serem carregados. 
                 Útil para delimitar um subconjunto menor durante testes rápidos.
    
    Returns:
        tuple: Uma tupla contendo 4 elementos:
            - dataset (list): Lista com os primeiros 'm' pares de datas (formato humano, formato de máquina).
            - human_vocab (dict): Dicionário que mapeia os caracteres das datas humanas para índices numéricos.
            - machine_vocab (dict): Dicionário que mapeia os caracteres das datas de máquina para índices numéricos.
            - inv_machine_vocab (dict): Dicionário reverso que mapeia os índices de volta para os caracteres de máquina.
    """

    caminho_dataset = "data/dataset.pkl"
    caminho_human_vocab = "data/human_vocab.pkl"
    caminho_machine_vocab = "data/machine_vocab.pkl"
    caminho_inv_machine_vocab = "data/inv_machine_vocab.pkl"

    with open(caminho_dataset, "rb") as f:
        dataset = pickle.load(f)
    dataset = dataset[:m]

    with open(caminho_human_vocab, "rb") as f:
        human_vocab = pickle.load(f)

    with open(caminho_machine_vocab, "rb") as f:
        machine_vocab = pickle.load(f)

    with open(caminho_inv_machine_vocab, "rb") as f:
        inv_machine_vocab = pickle.load(f)

    return dataset, human_vocab, machine_vocab, inv_machine_vocab

In [ ]:
m = 10000
dataset, human_vocab, machine_vocab, inv_machine_vocab = load_dataset(m)

In [ ]:
dataset[:10]

In [ ]:
machine_vocab

In [ ]:
def preprocess_data(dataset, human_vocab, machine_vocab, Tx, Ty):
    """
    Converte as strings de datas em tensores (numéricos e one-hot)
    com os tamanhos exatos exigidos pela rede neural.
    """
    X_numerico = []
    Y_numerico = []
    
    for data_h, data_m in dataset:
        # Pega o índice do caractere. Se não existir, pega o índice de '<unk>' (unknown) ou 0
        x_indices = [human_vocab.get(char.lower(), human_vocab.get('<unk>', 0)) for char in data_h]
        y_indices = [machine_vocab.get(char, 0) for char in data_m]
        
        X_numerico.append(x_indices)
        Y_numerico.append(y_indices)
    

    # Transforma listas em matrizes de tamanho (m, Tx) e (m, Ty)
    pad_char = human_vocab.get('<pad>', 36)
    X = pad_sequences(X_numerico, maxlen=Tx, padding='post', value=pad_char)
    Y = pad_sequences(Y_numerico, maxlen=Ty, padding='post', value=0)
    

    tamanho_vocab_humano = len(human_vocab)
    tamanho_vocab_maquina = len(machine_vocab)
    #Crias os vetores de one-hot
    Xoh = np.array([to_categorical(i, num_classes=tamanho_vocab_humano) for i in X])
    Yoh = np.array([to_categorical(i, num_classes=tamanho_vocab_maquina) for i in Y])
    
    return X, Y, Xoh, Yoh

In [ ]:
Tx = 30
Ty = 10
X, Y, Xoh, Yoh = preprocess_data(dataset, human_vocab, machine_vocab, Tx, Ty)

print("X.shape:", X.shape)
print("Y.shape:", Y.shape)
print("Xoh.shape:", Xoh.shape)
print("Yoh.shape:", Yoh.shape)

In [ ]:
index = 0
print("Source date:", dataset[index][0])
print("Target date:", dataset[index][1])
print()
print("Source after preprocessing (indices):", X[index])
print("Target after preprocessing (indices):", Y[index])
print()
print("Source after preprocessing (one-hot):", Xoh[index])
print("Target after preprocessing (one-hot):", Yoh[index])

<a name='2'></a>
## 2 - Neural Machine Translation with Attention

<a name='2-1'></a>
### 2.1 - Attention Mechanism

In [ ]:
repetidor = RepeatVector(Tx)
concatenador = Concatenate(axis=-1)
densa_1 = Dense(10, activation="tanh") 
densa_2 = Dense(1, activation="relu") 
ativacao_softmax = Softmax(axis=1, name='attention_weights') 
calculo_contexto = Dot(axes=1)

In [ ]:


n_a = 32 # number of units for the pre-attention, bi-directional LSTM's hidden state 'a'
n_s = 64 # number of units for the post-attention LSTM's hidden state "s"

post_activation_LSTM_cell = LSTM(n_s, return_state = True) 
output_layer = Dense(len(machine_vocab), activation="softmax")


def um_passo_de_atencao(a, s_prev):
    """
    Executa um passo de atenção para calcular o vetor de contexto.
    
    a: Saídas escondidas da Bi-LSTM (codificador)
    s_prev: Estado escondido anterior da LSTM (decodificador)
    """

    s_prev_repetido = repetidor(s_prev)
    
    concatenado = concatenador([a, s_prev_repetido])
    
    e = densa_1(concatenado)

    energias = densa_2(e)

    alphas = ativacao_softmax(energias)
    
    contexto = calculo_contexto([alphas, a])
    
    return contexto


In [ ]:

def modelf(Tx, Ty, n_a, n_s, human_vocab_size, machine_vocab_size):
    """
    Arguments:
    Tx -- length of the input sequence
    Ty -- length of the output sequence
    n_a -- hidden state size of the Bi-LSTM
    n_s -- hidden state size of the post-attention LSTM
    human_vocab_size -- size of the python dictionary "human_vocab"
    machine_vocab_size -- size of the python dictionary "machine_vocab"

    Returns:
    model -- Keras model instance
    """

    # Define the inputs of your model with a shape (Tx,)
    # Define s0 (initial hidden state) and c0 (initial cell state)
    # for the decoder LSTM with shape (n_s,)
    X = Input(shape=(Tx, human_vocab_size))
    s0 = Input(shape=(n_s,), name='s0')
    c0 = Input(shape=(n_s,), name='c0')
    s = s0
    c = c0

    # Initialize empty list of outputs
    outputs = []

    # Step 1: Define your pre-attention Bi-LSTM.
    a = Bidirectional(LSTM(n_a, return_sequences=True))(X)

    # Step 2: Iterate for Ty steps
    for t in range(Ty):

        # Step 2.A: Perform one step of the attention mechanism to get back the context vector at step t
        # context = one_step_attention(a, s)
        context = um_passo_de_atencao(a, s)

        # Step 2.B: Apply the post-attention LSTM cell to the "context" vector.
        # Don't forget to pass: initial_state = [hidden state, cell state]
        s, _, c = post_activation_LSTM_cell(context,initial_state=[s, c])

        # Step 2.C: Apply Dense layer to the hidden state output of the post-attention LSTM
        out = output_layer(s)

        # Step 2.D: Append "out" to the "outputs" list
        outputs.append(out)

    # Step 3: Create model instance taking three inputs and returning the list of outputs.
    model = Model(inputs=[X, s0, c0],outputs=outputs)

    return model

In [ ]:
model = modelf(Tx, Ty, n_a, n_s, len(human_vocab), len(machine_vocab))

In [ ]:
model.summary()

In [ ]:
opt = Adam(learning_rate=0.005, beta_1=0.9, beta_2=0.999, weight_decay=0.01)
model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy']*10)

In [ ]:
s0 = np.zeros((m, n_s))
c0 = np.zeros((m, n_s))
outputs = list(Yoh.swapaxes(0,1))

In [ ]:
# model.fit([Xoh, s0, c0], outputs, epochs=100, batch_size=100)
# model.save_weights("pesos100epocas.weights.h5")

In [ ]:
# Optional: loading pre-saved model
model.load_weights('pesos100epocas.weights.h5')

In [ ]:
def string_to_int(string, length, vocab):
    """
    Converts all strings in the vocabulary into a list of integers representing the positions of the
    input string's characters in the "vocab"
    
    Arguments:
    string -- input string, e.g. 'Wed 10 Jul 2007'
    length -- the number of time steps you'd like, determines if the output will be padded or cut
    vocab -- vocabulary, dictionary used to index every character of your "string"
    
    Returns:
    rep -- list of integers (or '<unk>') (size = length) representing the position of the string's character in the vocabulary
    """
    
    #make lower to standardize
    string = string.lower()
    string = string.replace(',','')
    
    if len(string) > length:
        string = string[:length]
        
    rep = list(map(lambda x: vocab.get(x, '<unk>'), string))
    
    if len(string) < length:
        rep += [vocab['<pad>']] * (length - len(string))
    
    return rep

In [ ]:
def translate_date(sentence):
    s00 = np.zeros((1, n_s))
    c00 = np.zeros((1, n_s))
    
    source = string_to_int(sentence, Tx, human_vocab)
    
    source = np.array(list(map(lambda x: to_categorical(x, num_classes=len(human_vocab)), source))).swapaxes(0,1)
    source = np.swapaxes(source, 0, 1)
    source = np.expand_dims(source, axis=0)
    
    prediction = model.predict([source, s00, c00])
    
    indices_vencedores = [np.argmax(p) for p in prediction]
    
    output = [inv_machine_vocab[i] for i in indices_vencedores]
    
    print("source:", sentence)
    print("output:", ''.join(output),"\n")
example = "4th of july 2001"
translate_date(example)

<a name='3'></a>
## 3 - Visualizing Attention (Optional / Ungraded)

<a name='3-1'></a>
### 3.1 - Getting the Attention Weights From the Network

In [ ]:
model.summary()

In [ ]:
def int_to_string(ints, inv_vocab):
    chars = []

    for i in ints:
        c = inv_vocab[i]

        if c == "<pad>":
            continue

        chars.append(c)

    return ''.join(chars)

In [ ]:
import numpy as np
# from string_to_int import string_to_int
# from int_to_string import int_to_string
import matplotlib.pyplot as plt
from keras.models import load_model, Model
from keras.utils import to_categorical


def plot_attention_map(modelx, input_vocabulary, inv_output_vocabulary, text, n_s = 128, num = 7):
    """
    Plot the attention map.
  
    """
    attention_map = np.zeros((10, 30))

    Ty, Tx = attention_map.shape
    
    human_vocab_size = 37
    
    # Well, this is cumbersome but this version of tensorflow-keras has a bug that affects the 
    # reuse of layers in a model with the functional API. 
    # So, I have to recreate the model based on the functional 
    # components and connect then one by one.
    # ideally it can be done simply like this:
    # layer = modelx.layers[num]
    # f = Model(modelx.inputs, [layer.get_output_at(t) for t in range(Ty)])
    #
    
    X = modelx.inputs[0] 
    s0 = modelx.inputs[1] 
    c0 = modelx.inputs[2] 
    s = s0
    c = c0
    
    a = modelx.layers[2](X)  
    outputs = []

    for t in range(Ty):
        s_prev = s
        s_prev = modelx.layers[3](s_prev)
        concat = modelx.layers[4]([a, s_prev]) 
        e = modelx.layers[5](concat) 
        energies = modelx.layers[6](e) 
        alphas = modelx.layers[7](energies) 
        context = modelx.layers[8]([alphas, a])
        # Don't forget to pass: initial_state = [hidden state, cell state] (≈ 1 line)
        s, _, c = modelx.layers[10](context, initial_state = [s, c]) 
        outputs.append(alphas)

    f = Model(inputs=[X, s0, c0], outputs = outputs)
    

    s0 = np.zeros((1, n_s))
    c0 = np.zeros((1, n_s))
    encoded = np.array(string_to_int(text, Tx, input_vocabulary)).reshape((1, 30))
    encoded = np.array(list(map(lambda x: to_categorical(x, num_classes=len(input_vocabulary)), encoded)))

    
    r = f([encoded, s0, c0])
        
    for t in range(Ty):
        for t_prime in range(Tx):
            attention_map[t][t_prime] = r[t][0, t_prime,0]

    input_length = len(text)

    # 1. Normalizar PRIMEIRO (isso garante que o pico real no padding seja o 1.0)
    print(attention_map[4:10, :input_length])
    
    row_max = attention_map.max(axis=1)
    attention_map = attention_map / np.where(row_max == 0, 1, row_max)[:, None]

    # 2. Cortar DEPOIS (agora as linhas finais terão valores próximos a 0.0)
    attention_map = attention_map[:, :input_length]

    prediction = modelx.predict([encoded, s0, c0])
    
    predicted_text = []
    for i in range(len(prediction)):
        predicted_text.append(int(np.argmax(prediction[i], axis=1)[0]))
        
    predicted_text = list(predicted_text)
    predicted_text = int_to_string(predicted_text, inv_output_vocabulary)
    text_ = list(text)
    
    # get the lengths of the string
    input_length = len(text)
    output_length = Ty
    
    # Plot the attention_map
    plt.clf()
    f = plt.figure(figsize=(8, 8.5))
    ax = f.add_subplot(1, 1, 1)

    # add image
    i = ax.imshow(attention_map, interpolation='nearest', cmap='Blues')

    # add colorbar
    cbaxes = f.add_axes([0.2, 0, 0.6, 0.03])
    cbar = f.colorbar(i, cax=cbaxes, orientation='horizontal')
    cbar.ax.set_xlabel('Alpha value (Probability output of the "softmax")', labelpad=2)

    # add labels
    ax.set_yticks(range(output_length))
    ax.set_yticklabels(predicted_text[:output_length])

    ax.set_xticks(range(input_length))
    ax.set_xticklabels(text_[:input_length], rotation=45)

    ax.set_xlabel('Input Sequence')
    ax.set_ylabel('Output Sequence')

    # add grid and legend
    ax.grid()

    plt.show()
    
    return attention_map



In [ ]:
attention_map = plot_attention_map(model, human_vocab, inv_machine_vocab, "Tuesday 09 Oct 1993", num = 7, n_s = 64);